# Modelos de árboles y gradient boosting — 6 meses (enero-junio 2025)

Semana 10 del plan de trabajo (proposal.md, sección 13). Entrenamos tres modelos basados en árboles sobre el mismo pipeline de features del notebook 02 y los comparamos contra el mejor modelo lineal (Ridge, MAE = 5.14 min en mayo):

1. **Random Forest** con profundidad y muestreo acotados (riesgo de memoria señalado en proposal.md, sección 12).
2. **HistGradientBoostingRegressor** de scikit-learn.
3. **LightGBM**.

Los hiperparámetros quedan fijos en valores razonables; la búsqueda con validación temporal corresponde a la semana 11. El conjunto de test (junio) no se toca.

In [1]:
import sys
sys.path.append('..')

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_log_error

from src.data import cargar_todo, split_temporal, MESES
from src.features import construir_preprocesador, preparar_xy

SEED = 42
np.random.seed(SEED)

import src.features as features
features.RUTA_ZONAS = Path('../data/Taxi_Zone_Lookup.csv')
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

## 1. Carga de los 6 meses y split temporal
Mismo muestreo que el notebook 02 (`SAMPLE_FRAC = 0.20`, `SEED = 42`) para que las métricas sean comparables.

In [2]:
SAMPLE_FRAC = 0.20

df = cargar_todo(meses=MESES, sample_frac=SAMPLE_FRAC, seed=SEED)
train, val, test = split_temporal(df)
del df
print(f"Train (ene-abr): {len(train):,}")
print(f"Val   (mayo):    {len(val):,}")
print(f"Test  (junio):   {len(test):,}  <- no se usa en este notebook")

Train (ene-abr): 2,872,441
Val   (mayo):    855,412
Test  (junio):   803,222  <- no se usa en este notebook


## 2. Features

Usamos `construir_preprocesador(escalar_numericas=False)`: los árboles no necesitan escalado. El preprocesador se ajusta **una sola vez y solo con train** (incluye la distancia histórica por par de zonas y el TargetEncoder de zonas) y luego se aplica a validación. Es equivalente a meterlo en cada `Pipeline`, pero evita reajustar el TargetEncoder (cv=5 sobre ~2.9M filas) para cada modelo.

**Target en escala log.** La duración tiene cola larga (notebook 01, visualización 1). Entrenamos los tres modelos sobre `log1p(duración)` y deshacemos la transformación con `expm1` al predecir. Esto optimiza directamente el error relativo (RMSLE) y, al ser la media en escala log cercana a la mediana, también favorece el MAE.

In [3]:
X_tr, y_tr_log = preparar_xy(train, usar_log_target=True)
X_val, _ = preparar_xy(val)
y_val = val['trip_duration_min'].to_numpy(dtype=float)

t0 = time.time()
prep = construir_preprocesador(escalar_numericas=False)
Xt_tr = prep.fit_transform(X_tr, y_tr_log).astype(np.float32)
Xt_val = prep.transform(X_val).astype(np.float32)
nombres_features = prep.named_steps['codificador'].get_feature_names_out()
del X_tr, X_val, train, val, test
print(f"Preprocesamiento: {time.time() - t0:.0f} s | {Xt_tr.shape[1]} features")

Preprocesamiento: 9 s | 41 features


## 3. Función de evaluación
MAE (principal) y RMSLE (secundaria), igual que en el notebook 02. Además del score, registramos el tiempo de entrenamiento.

In [4]:
def evaluar(nombre, y_real, y_pred, resultados, segundos=None):
    y_pred = np.clip(y_pred, 0, None)
    mae = mean_absolute_error(y_real, y_pred)
    rmsle = np.sqrt(mean_squared_log_error(y_real, y_pred))
    print(f"{nombre:<42} MAE = {mae:6.2f} min | RMSLE = {rmsle:.4f}")
    resultados.append({'modelo': nombre, 'MAE': mae, 'RMSLE': rmsle, 'segundos_entrenamiento': segundos})
    return resultados


def entrenar_y_evaluar(nombre, modelo, resultados):
    t0 = time.time()
    modelo.fit(Xt_tr, y_tr_log)
    segundos = round(time.time() - t0, 1)
    pred = np.expm1(modelo.predict(Xt_val))
    print(f"  (entrenamiento: {segundos} s)")
    return evaluar(nombre, y_val, pred, resultados, segundos)


resultados = []

## 4. Random Forest

Con `max_depth=None` cada árbol crecería hasta hojas de 1 viaje sobre ~2.9M filas y no cabría en memoria. Lo acotamos con:
- `max_samples=0.1`: cada árbol ve un bootstrap de ~290k filas.
- `max_depth=20` y `min_samples_leaf=50`: árboles de tamaño controlado.
- `max_features=0.5`: decorrelaciona los árboles.

In [5]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=50,
    max_features=0.5,
    max_samples=0.1,
    n_jobs=-1,
    random_state=SEED,
)
resultados = entrenar_y_evaluar("F: Random Forest", rf, resultados)

  (entrenamiento: 16.3 s)
F: Random Forest                           MAE =   4.40 min | RMSLE = 0.3368


## 5. HistGradientBoostingRegressor
Discretiza cada feature en 255 bins, lo que lo hace viable sobre millones de filas. Sin early stopping: el número de iteraciones queda fijo para no usar mayo como criterio de parada.

In [6]:
hgb = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.1,
    max_leaf_nodes=63,
    min_samples_leaf=100,
    early_stopping=False,
    random_state=SEED,
)
resultados = entrenar_y_evaluar("G: HistGradientBoosting", hgb, resultados)

  (entrenamiento: 62.9 s)
G: HistGradientBoosting                    MAE =   4.13 min | RMSLE = 0.3233


## 6. LightGBM
Configuración comparable a la de HistGradientBoosting (mismo número de iteraciones y learning rate), con más hojas por árbol y submuestreo de filas y columnas.

In [7]:
lgbm = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.1,
    num_leaves=127,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=SEED,
    verbose=-1,
)
resultados = entrenar_y_evaluar("H: LightGBM", lgbm, resultados)

  (entrenamiento: 20.6 s)
H: LightGBM                                MAE =   4.07 min | RMSLE = 0.3206


## 7. Comparación con los baselines del notebook 02
Unimos estas métricas con las de `outputs/metrics_6meses.json` (mismo split, mismo conjunto de validación). Del barrido de Ridge solo mostramos el modelo D.

In [8]:
base = pd.DataFrame(json.load(open('../outputs/metrics_6meses.json', encoding='utf-8')))
base = base[base['modelo'].str.match(r'^[A-E]:')]

tabla = (
    pd.concat([base, pd.DataFrame(resultados)], ignore_index=True)
    .sort_values('MAE')
    .reset_index(drop=True)
)
mejor_lineal = base.loc[base['MAE'].idxmin(), 'MAE']
tabla['mejora_vs_lineal_%'] = ((mejor_lineal - tabla['MAE']) / mejor_lineal * 100).round(1)
tabla

,modelo,MAE,RMSLE,segundos_entrenamiento,mejora_vs_lineal_%
0,H: LightGBM,4.072227,0.320563,20.6,20.8
1,G: HistGradientBoosting,4.126053,0.323255,62.9,19.8
2,F: Random Forest,4.396556,0.336791,16.3,14.5
3,"D: Ridge (regularizada, alpha=1.0)",5.143940,0.388650,NaN,0.0
4,C: Regresión lineal (sin regularizar),5.143942,0.388651,NaN,-0.0
5,"E: Lasso (regularizada, alpha=0.01)",5.144750,0.387012,NaN,-0.0
6,"B: Mediana por (zona, hora)",7.858684,0.580354,NaN,-52.8
7,A: Mediana global,9.283539,0.664978,NaN,-80.5


**Chequeo de sanidad (proposal.md, sección 8):** si algún modelo baja de 2 min de MAE, se audita la lista de features antes de reportarlo.

In [9]:
assert tabla['MAE'].min() >= 2, "MAE sospechosamente bajo: auditar features por leakage"
print("OK: ningún modelo baja de 2 min de MAE")

OK: ningún modelo baja de 2 min de MAE


## 8. Importancia de features (LightGBM)
Chequeo rápido de qué está usando el mejor modelo de boosting (ganancia total por feature). La interpretabilidad completa (SHAP) queda para la semana 13.

In [10]:
importancia = (
    pd.Series(lgbm.booster_.feature_importance(importance_type='gain'), index=nombres_features)
    .sort_values(ascending=False)
)
(importancia / importancia.sum() * 100).round(2).head(15).to_frame('% de la ganancia')

,% de la ganancia
num__distancia_estimada_mi,76.82
num__log_distancia_estimada,8.83
target_enc__DOLocationID,2.69
target_enc__PULocationID,2.36
num__pickup_hour,1.98
num__hora_cos,1.97
num__pickup_dayofweek,0.86
num__hora_sin,0.69
num__dia_sin,0.51
num__viaje_aeropuerto,0.45


In [11]:
Path('../outputs').mkdir(exist_ok=True)
with open('../outputs/metrics_arboles.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print("Guardado en ../outputs/metrics_arboles.json")

Guardado en ../outputs/metrics_arboles.json


## 9. Lectura de resultados

Resultados sobre validación (mayo), mismo split y muestreo que el notebook 02:

| Modelo | MAE (min) | RMSLE | Entrenamiento |
|---|---|---|---|
| A. Mediana global | 9.28 | 0.665 | — |
| B. Mediana por (zona, hora) | 7.86 | 0.580 | — |
| D. Ridge (mejor lineal) | 5.14 | 0.389 | — |
| F. Random Forest | 4.40 | 0.337 | 16 s |
| G. HistGradientBoosting | 4.13 | 0.323 | 63 s |
| **H. LightGBM** | **4.07** | **0.321** | 21 s |

**Los árboles justifican su complejidad.** Los tres superan al mejor modelo lineal: LightGBM baja el MAE de 5.14 a 4.07 min (−21 %) y el RMSLE de 0.389 a 0.321. Los dos modelos de boosting quedan prácticamente empatados (diferencia de 0.05 min). LightGBM entrena 3 veces más rápido, así que es el candidato natural para la búsqueda de hiperparámetros. Random Forest queda atrás: con `max_samples=0.1` y `max_depth=20` está deliberadamente acotado por memoria.

**Matiz en la comparación.** Los modelos de este notebook se entrenan sobre `log1p(duración)` y los del notebook 02 sobre la duración en minutos, así que parte de la mejora puede venir del cambio de target y no solo del tipo de modelo. Para separar ambos efectos habría que reentrenar Ridge sobre el target en log.

**Qué usa el modelo.** La distancia estimada (mediana histórica por par de zonas, aprendida solo en train) concentra ~86 % de la ganancia entre sus dos versiones. Le siguen el target encoding de destino y origen (~5 %) y la hora del día (~5 % entre `pickup_hour`, `hora_sin` y `hora_cos`). Ningún modelo baja de 2 min de MAE, así que no hay señales de leakage según el chequeo del proposal.

**Punto a revisar: `pickup_month`.** En train solo existen los meses 1 a 4, así que para mayo (y junio en test) los árboles tratan el mes como si fuera abril: es una feature que no puede generalizar con split temporal. Su importancia es baja (0.2 % de la ganancia), pero conviene evaluar quitarla en la semana 11.

**Siguientes pasos (semana 11):** búsqueda de hiperparámetros de LightGBM (y opcionalmente HistGradientBoosting) con `TimeSeriesSplit` dentro de enero-abril, como indica la sección 10 del proposal. Mayo queda solo para comparar las configuraciones finales y junio se toca una sola vez al final.

Las métricas de este notebook están en `outputs/metrics_arboles.json`.